# 1. Data Ingestion

AI Events, Incidents and Market data is pulled into the data folder for analysis
Sources:
1. **AIID** (AI Incident Database) - structured AI incident records(date, title, harm_category, affected_parties)
2. **Forecast-Dojo** (Events and Market Data) - Polymarket events, dated news articles, forecast steps
3. **FinKario** (Financial Knowledge graph) - entities, relational triples, event-related stock insights

### Imports and Configuration

In [1]:
import os
import json
import ast
from pathlib import Path

import pandas as pd
import numpy as np
import yfinance as yf

RAW = Path("data/raw")
PROCESSED = Path("data/processed")
PROCESSED.mkdir(parents=True, exist_ok=True)

AIID_RAW = RAW / "aiid"

HF_DATASET_ID = "FinEredium1/Forecast-Dojo/data"

print(f"Processed data: {PROCESSED}")


Processed data: data\processed


In [2]:
%pwd

'c:\\Users\\Koushik Sripathi\\Desktop\\AI-Event-Market-Prediction\\notebooks'

In [3]:
os.chdir("..")
%pwd

'c:\\Users\\Koushik Sripathi\\Desktop\\AI-Event-Market-Prediction'

### 1. Load AIID Incident Data

The AIID snapshot is a mongodump. The flat CSV lives under `mongodump_full_snapshot/`.
We use a recursive search so the loader survives re-downloads with different folder names.

In [4]:
def find_file(base: Path, name: str) -> Path:
    matches = list(base.rglob(name))
    if not matches:
        raise FileNotFoundError(f"Could not find {name} under {base.resolve()}")
    return matches[0]


def load_incidents() -> pd.DataFrame:
    path = find_file("incidents.csv")
    print(f"Loading {path}")
    df = pd.read_csv(path)
    print(f"Columns: {df.columns.tolist()}")
    return df

In [ ]:
data\raw\aiid\mongodump_full_snapshot\incidents.csv

In [5]:
def load_aiid_incidents() -> pd.DataFrame:
    path = find_file(AIID_RAW , "incidents.csv")
    print(f"Loading AIID from: {path}")

    df = pd.read_csv(path)
    print(f"Rows: {len(df)}")
    print(f"Columns: {df.columns.tolist()}")

    # Normalise date column
    if "date" in df.columns:
        df["date"] = pd.to_datetime(df["date"], errors="coerce", utc=True)
        df = df.sort_values("date").reset_index(drop=True)

    return df


aiid_df = load_aiid_incidents()
aiid_df.head(3)

Loading AIID from: data\raw\aiid\mongodump_full_snapshot\incidents.csv
Rows: 1703
Columns: ['_id', 'incident_id', 'date', 'reports', 'Alleged deployer of AI system', 'Alleged developer of AI system', 'Alleged harmed or nearly harmed parties', 'description', 'title']


,_id,incident_id,date,reports,Alleged deployer of AI system,Alleged developer of AI system,Alleged harmed or nearly harmed parties,description,title
0,ObjectId(625763df343edc875fe63a19),27,1983-09-26 00:00:00+00:00,"[342,343,344,345,346,347,349,350,351,352,353,3...","[""soviet-union""]","[""soviet-union""]","[""all-life-on-earth""]",An alert of five incoming intercontinental bal...,Nuclear False Alarm
1,ObjectId(636218985a33233a22f6632e),379,1992-05-25 00:00:00+00:00,"[2179,2180]","[""pepsi""]","[""d.g.-consultores""]","[""filipinos""]",Pepsi's number generation system determining d...,Error in Pepsi's Number Generation System Led ...
2,ObjectId(625763e0343edc875fe63a28),42,1996-04-03 00:00:00+00:00,"[759,2471]","[""national-resident-matching-program""]","[""national-resident-matching-program""]","[""medical-residents""]","Alvin Roth, a Ph.D at the University of Pittsb...",Inefficiencies in the United States Resident M...


In [8]:
# AIID summary
print(f"Date range: {aiid_df['date'].min()} - {aiid_df['date'].max()}")
print(f"Shape of the dataset: {aiid_df.shape}")
print(f"No. of missing entries: \n{aiid_df.isna().sum().sort_values(ascending=False)}")

# Save a lightwight processed version
aiid_df.to_parquet(PROCESSED / "aiid_df.parquet", index=False)
print(f"\nSaved {len(aiid_df)} incidents → {PROCESSED / 'aiid_incidents.parquet'}")

Date range: 1983-09-26 00:00:00+00:00 - 2026-09-25 00:00:00+00:00
Shape of the dataset: (1703, 9)
No. of missing entries: 
_id                                        0
incident_id                                0
date                                       0
reports                                    0
Alleged deployer of AI system              0
Alleged developer of AI system             0
Alleged harmed or nearly harmed parties    0
description                                0
title                                      0
dtype: int64

Saved 1703 incidents → data\processed\aiid_incidents.parquet


### 2. Load Market Data (yfinance)

We download daily Open, High, Low, Close and Volume (OHLCV) for a set of AI-related tickers. These are the stocks most
likely to react to AI events (model releases, incidents, regulation).

Extend `TICKERS` as needed — for Indian markets use the `.NS` suffix (e.g. `RELIANCE.NS`).